# 06 — Traditional ML Anomaly Detection

## 1. Objective

This notebook implements and investigates three traditional machine-learning methods for detecting anomalous behaviour in the SWaT multivariate time-series dataset:

1. **Isolation Forest**
2. **One-Class Support Vector Machine (One-Class SVM)**
3. **Local Outlier Factor (LOF)** using novelty-detection mode.

The objective is to establish how these methods identify observations that differ from patterns learned from normal operating data. Their assumptions, preprocessing requirements, computational costs, anomaly scores, and detection behaviour will be documented and assessed.

## 2. Scope

Each method will be investigated separately using a consistent experimental design wherever technically appropriate. The implementation will preserve chronological data boundaries, fit learned preprocessing on permitted training data, and avoid using attack labels as model features or for selecting thresholds on the final test set.

Particular attention will be given to feature scaling, model hyperparameters, anomaly-score orientation, threshold selection, and computational feasibility.

The results will be documented after each experiment. No method will be considered superior without supporting evidence from the evaluation.

This notebook complements the three baseline methods in Notebook 04 and the LSTM Autoencoder in Notebook 05. Its results will contribute to the subsequent unified evaluation of all nine selected detection methods.

## 3. Methodological Foundations

### 3.1 Learning from Normal Operating Data

The three detectors in this notebook are trained using normal operating data only. Their purpose is to identify observations that differ from the patterns learned during training.

Let the training dataset be

$$
X_{\mathrm{train}} = \{x_1, x_2, \ldots, x_n\}
$$

where each observation $x_i \in \mathbb{R}^{d}$ contains $d$ sensor features.

Each detector learns a representation of normal operating behaviour and assigns an anomaly score to an observation. The score is converted into a binary decision using a threshold:

$$
\hat{y}_i =
\begin{cases}
1, & s(x_i) > \tau, \\
0, & s(x_i) \leq \tau,
\end{cases}
$$

where $s(x_i)$ is the anomaly score, $\tau$ is the selected threshold, $1$ denotes a flagged observation, and $0$ denotes an observation that is not flagged.

The score direction must be verified for each implementation, because some estimators assign lower scores to more anomalous observations and others assign higher scores. Applying a fixed threshold without confirming the direction will produce inverted decisions.

A flagged observation is a model-generated anomaly alert. It is not, by itself, proof of a cyberattack or physical fault.

### 3.2 Isolation Forest

Isolation Forest identifies observations that can be isolated from the rest of the data using randomly selected features and split values.

The underlying principle is that observations that differ substantially from the broader data distribution may require fewer splits to isolate. Their average path lengths in the isolation trees therefore tend to be shorter than those of normal observations.

The method is controlled by parameters such as the number of trees, the number of observations sampled for each tree, and the contamination setting.

The contamination parameter affects the estimator's internal classification boundary. It must not be confused with a threshold calibrated independently using permitted normal calibration data. The contamination setting is a property of the fitted estimator, while the final anomaly threshold is a decision rule applied afterwards to the anomaly scores.

### 3.3 One-Class Support Vector Machine

One-Class SVM learns a decision boundary that describes the training-data distribution. Observations outside the learned region can be identified as potential anomalies.

Its behaviour depends on the kernel, the $\nu$ parameter, and feature scaling.

The parameter $\nu$ provides an upper bound on the fraction of training errors and a lower bound on the fraction of support vectors under the model formulation. It is not a guarantee of the detector's false-positive rate on future data. In particular, a small $\nu$ does not guarantee that approximately $\nu$ of future normal observations will be flagged; the empirical rate depends on how representative the training period is of the deployment period.

Because sensor features can have different numerical ranges, the scaling procedure must be examined carefully. Any fitted scaler must be fitted on training data only and applied to later partitions without refitting.

One-Class SVM can also become computationally expensive as the number of training observations increases, with typical complexity scaling between linear and quadratic in the sample size depending on the kernel. The training sample size and runtime must therefore be considered before selecting an implementation.

### 3.4 Local Outlier Factor

Local Outlier Factor identifies observations whose local density differs from that of their neighbours.

The method compares an observation's local reachability density with the corresponding densities of its nearest neighbours. An observation with substantially lower local density than its neighbours receives a higher outlier score.

The neighbourhood size $k$ influences the scale at which local differences are detected. Small neighbourhoods can be sensitive to local fluctuations, while larger neighbourhoods may obscure smaller-scale structures and reduce sensitivity to local anomalies.

Scikit-learn distinguishes between ordinary outlier detection and novelty detection. Ordinary outlier detection (`novelty=False`) assigns scores to the same observations used to fit the model, and cannot be applied to previously unseen data. Novelty detection (`novelty=True`) fits the model on training data and scores new observations as they arrive. Because this project trains detectors on one period and applies them to previously unseen observations, LOF must use `novelty=True`. This allows the fitted model to score new observations without treating the evaluation data as part of the original neighbourhood search.

### 3.5 Chronology, Preprocessing, and Leakage Control

SWaT is a time-series dataset, so chronological order must be preserved. Randomly mixing observations across training and evaluation periods could make the experiment less representative of detecting previously unseen behaviour, and could allow later-period statistics to influence models that are supposed to have been trained on earlier data alone.

All learned preprocessing must be fitted using the permitted training data and then applied to later partitions without refitting.

The models must not receive the `Normal/Attack` label as an input feature. Attack labels must also not be used to optimise thresholds on the final test set.

Threshold calibration and final evaluation follow the project's established evaluation protocol, with the calibration/threshold/validation separation used in earlier notebooks. The implementation details and any method-specific deviations are documented in the relevant sections.

In [ ]:
from pathlib import Path

import joblib
import numpy as np

# Confirmed Google Drive locations
DRIVE_ROOT = Path("/content/drive/MyDrive")
DATA_DIR = DRIVE_ROOT / "swat-dataset"
ARTIFACT_DIR = DRIVE_ROOT / "swat-artifacts"

NORMAL_PATH = DATA_DIR / "normal.csv"
SCALER_PATH = ARTIFACT_DIR / "scaler.joblib"
SEQUENCE_PATH = ARTIFACT_DIR / "evaluation_sequences.npz"
MODEL_PATH = ARTIFACT_DIR / "lstm_autoencoder.pth"

# 1. Verify relevant files
print("=" * 60)
print("FILE VERIFICATION")
print("=" * 60)

for path in (NORMAL_PATH, SCALER_PATH, SEQUENCE_PATH, MODEL_PATH):
    print(f"{path.name}: {'FOUND' if path.is_file() else 'NOT FOUND'}")
    print(f"  Path: {path}")

if not NORMAL_PATH.is_file():
    raise FileNotFoundError(f"Normal dataset not found: {NORMAL_PATH}")

if not SCALER_PATH.is_file():
    raise FileNotFoundError(f"Scaler not found: {SCALER_PATH}")

if not SEQUENCE_PATH.is_file():
    raise FileNotFoundError(f"Evaluation sequences not found: {SEQUENCE_PATH}")

# 2. Inspect the existing scaler
print("\n" + "=" * 60)
print("SCALER INSPECTION")
print("=" * 60)

scaler = joblib.load(SCALER_PATH)

print(f"Scaler class: {type(scaler).__name__}")
print(f"Number of input features: {getattr(scaler, 'n_features_in_', 'Unavailable')}")

if hasattr(scaler, "feature_names_in_"):
    print(f"Feature names: {list(scaler.feature_names_in_)}")
else:
    print("Feature names: Not stored in the scaler.")

for attribute in ("data_min_", "data_max_", "data_range_", "scale_", "min_"):
    if hasattr(scaler, attribute):
        values = np.asarray(getattr(scaler, attribute))
        print(f"{attribute}: shape={values.shape}")
        print(f"  First 10 values: {values[:10]}")

# 3. Confirm the saved sequence structure
print("\n" + "=" * 60)
print("SAVED SEQUENCE STRUCTURE")
print("=" * 60)

with np.load(SEQUENCE_PATH, allow_pickle=False) as artifacts:
    for name in artifacts.files:
        array = artifacts[name]
        print(f"{name}: shape={array.shape}, dtype={array.dtype}")

print("\nInspection complete. No training or CSV loading was performed.")

FILE VERIFICATION
normal.csv: FOUND
  Path: /content/drive/MyDrive/swat-dataset/normal.csv
scaler.joblib: FOUND
  Path: /content/drive/MyDrive/swat-artifacts/scaler.joblib
evaluation_sequences.npz: FOUND
  Path: /content/drive/MyDrive/swat-artifacts/evaluation_sequences.npz
lstm_autoencoder.pth: FOUND
  Path: /content/drive/MyDrive/swat-artifacts/lstm_autoencoder.pth

SCALER INSPECTION
Scaler class: MinMaxScaler
Number of input features: 42
Feature names: ['FIT101', 'LIT101', 'MV101', 'P101', 'P102', 'AIT201', 'AIT202', 'AIT203', 'FIT201', 'MV201', 'P201', 'P203', 'P205', 'DPIT301', 'FIT301', 'LIT301', 'MV301', 'MV302', 'MV303', 'MV304', 'P302', 'AIT401', 'AIT402', 'FIT401', 'LIT401', 'P402', 'P403', 'UV401', 'AIT501', 'AIT502', 'AIT503', 'AIT504', 'FIT501', 'FIT502', 'FIT503', 'FIT504', 'P501', 'PIT501', 'PIT502', 'PIT503', 'FIT601', 'P602']
data_min_: shape=(42,)
  First 10 values: [  0.     443.0066   0.       1.       1.     187.7083   6.     308.3825
   0.       0.    ]
data_max_:

## 4. Data and Input Representation

The traditional machine-learning methods in this notebook will operate on the same 42 sensor features established during the LSTM Autoencoder stage.

The feature set is:

\[
X \in \mathbb{R}^{n \times 42}
\]

where each row represents one chronological observation and each column represents one retained sensor or actuator feature.

The previously established feature-selection and scaling decisions will be reused rather than recomputed independently in this notebook. The saved `MinMaxScaler` contains the 42-feature preprocessing fitted during the earlier normal-data training stage.

The traditional detectors will therefore receive scaled 42-dimensional observations rather than the 30-step sequence tensors used by the LSTM Autoencoder.

This distinction is intentional. The LSTM Autoencoder models temporal sequences explicitly, whereas Isolation Forest, One-Class SVM, and Local Outlier Factor will first be evaluated using individual multivariate observations.

### 4.1 Training Data

Model fitting will use normal operating observations only.

Attack-labelled observations will not be used as training inputs, and attack labels will not be used to select model hyperparameters or anomaly thresholds.

The chronological structure established in the earlier notebooks will be preserved. No random mixing of observations across temporal partitions will be introduced.

### 4.2 Scaling

The existing `MinMaxScaler` will be reused because it was fitted during the established normal-data preprocessing stage.

For a feature \(x_j\), Min-Max scaling is represented as

\[
x'_j =
\frac{x_j - x_{j,\min}}
{x_{j,\max} - x_{j,\min}}
\]

where the minimum and maximum values were learned from the permitted normal training data.

The scaler will be applied to later observations without refitting.

This prevents the traditional ML experiments from introducing a second, independently fitted preprocessing scheme.

### 4.3 Method-Specific Considerations

The three methods will use the same 42-dimensional scaled observation representation where technically appropriate.

However, their computational requirements differ substantially:

- **Isolation Forest** can operate on a comparatively large normal training set.
- **One-Class SVM** can become computationally expensive as the number of training observations increases.
- **LOF** can also become computationally expensive because of its nearest-neighbour calculations and will use `novelty=True` for scoring previously unseen observations.

These computational constraints will be considered when defining the model-fitting sample. Any reduction in training observations must preserve chronological order and be explicitly documented rather than introduced implicitly.

The same established evaluation observations will subsequently be used to compare the detectors under a consistent protocol.